# Draft Helper

Startup notebook for loading hitter and pitcher leaderboards via pybaseball's built-in cache behavior and browsing them in tabbed grids.

In [1]:
from __future__ import annotations

from datetime import date
from typing import Dict, List

import pandas as pd
import ipywidgets as widgets
import ipydatagrid as dg
from IPython.display import display
from pybaseball import batting_stats, pitching_stats, cache

cache.enable()
print(cache.config.cache_directory)

/Users/colettace/.pybaseball/cache


In [2]:
CURRENT_YEAR = date.today().year
YEARS_TO_LOAD = [CURRENT_YEAR - 3, CURRENT_YEAR - 2, CURRENT_YEAR - 1]


def normalize_years(years: List[int]) -> List[int]:
    valid_years = sorted({year for year in years if 1871 <= year <= CURRENT_YEAR})
    if not valid_years:
        raise ValueError('No valid years provided.')
    return valid_years


def fetch_season_frames(year: int) -> Dict[str, pd.DataFrame]:
    """Use pybaseball API; pybaseball cache handles read/download automatically."""
    hitters_df = batting_stats(year, qual=0).copy()
    pitchers_df = pitching_stats(year, qual=0).copy()

    hitters_df['Season'] = year
    pitchers_df['Season'] = year
    return {'batting': hitters_df, 'pitching': pitchers_df}


def load_hitter_pitcher_frames(years: List[int]) -> Dict[str, pd.DataFrame]:
    all_batting: List[pd.DataFrame] = []
    all_pitching: List[pd.DataFrame] = []

    for year in normalize_years(years):
        season = fetch_season_frames(year)
        all_batting.append(season['batting'])
        all_pitching.append(season['pitching'])

    return {
        'hitters_raw': pd.concat(all_batting, ignore_index=True),
        'pitchers_raw': pd.concat(all_pitching, ignore_index=True),
    }


frames = load_hitter_pitcher_frames(YEARS_TO_LOAD)
hitters_raw = frames['hitters_raw']
pitchers_raw = frames['pitchers_raw']

print(f'Loaded hitter rows: {len(hitters_raw):,}')
print(f'Loaded pitcher rows: {len(pitchers_raw):,}')

Loaded hitter rows: 4,381
Loaded pitcher rows: 2,591


In [3]:
def normalize_core_columns(df: pd.DataFrame, player_type: str) -> pd.DataFrame:
    col_aliases = {
        'PlayerName': ['Name', 'name', 'player_name', 'player'],
        'PlayerID': ['IDfg', 'ID', 'playerid', 'player_id', 'mlb_id'],
        'Team': ['Team', 'Tm', 'team'],
        'Pos': ['Pos', 'position', 'Position'],
        'Season': ['Season', 'season', 'year', 'Year'],
    }

    normalized = pd.DataFrame(index=df.index)
    for canonical, aliases in col_aliases.items():
        normalized[canonical] = pd.NA
        for alias in aliases:
            if alias in df.columns:
                normalized[canonical] = df[alias]
                break

    if player_type == 'hitters':
        ranking_stats = ['HR', 'SB', 'R', 'RBI', 'AVG', 'OBP', 'SLG', 'wRC+', 'WAR']
        sort_cols = [c for c in ['WAR', 'HR', 'SB'] if c in df.columns]
    else:
        ranking_stats = ['W', 'SV', 'SO', 'K/9', 'ERA', 'WHIP', 'WAR']
        sort_cols = [c for c in ['WAR', 'SO', 'SV'] if c in df.columns]

    for col in ranking_stats:
        normalized[col] = df[col] if col in df.columns else pd.NA

    if sort_cols:
        normalized = normalized.sort_values(by=sort_cols, ascending=[False] * len(sort_cols), na_position='last')

    normalized = normalized.reset_index(drop=True)
    normalized.insert(0, 'Rank', normalized.index + 1)
    return normalized


hitters = normalize_core_columns(hitters_raw, player_type='hitters')
pitchers = normalize_core_columns(pitchers_raw, player_type='pitchers')

hitters.head(3), pitchers.head(3)

(   Rank      PlayerName  PlayerID Team   Pos  Season  HR  SB    R  RBI    AVG  \
 0     1     Aaron Judge     15640  NYY  -3.4    2024  58  10  122  144  0.322   
 1     2  Bobby Witt Jr.     25764  KCR   7.1    2024  32  31  125  109  0.332   
 2     3     Aaron Judge     15640  NYY -10.3    2025  53  12  137  114  0.331   
 
      OBP    SLG   wRC+   WAR  
 0  0.458  0.701  220.0  11.3  
 1  0.389  0.588  169.0  10.5  
 2  0.457  0.688  204.0  10.1  ,
    Rank    PlayerName  PlayerID Team   Pos  Season   W  SV   SO    K/9   ERA  \
 0     1  Tarik Skubal     22267  DET  <NA>    2025  13   0  241  11.10  2.21   
 1     2   Paul Skenes     33677  PIT  <NA>    2025  10   0  216  10.36  1.97   
 2     3    Chris Sale     10603  ATL  <NA>    2024  18   0  225  11.40  2.38   
 
    WHIP  WAR  
 0  0.89  6.6  
 1  0.95  6.5  
 2  1.01  6.4  )

In [4]:
hitter_grid = dg.DataGrid(
    hitters,
    selection_mode='row',
    base_row_size=28,
    layout=widgets.Layout(width='100%', height='650px'),
)

pitcher_grid = dg.DataGrid(
    pitchers,
    selection_mode='row',
    base_row_size=28,
    layout=widgets.Layout(width='100%', height='650px'),
)

tab = widgets.Tab(children=[hitter_grid, pitcher_grid])
tab.set_title(0, 'Hitters')
tab.set_title(1, 'Pitchers')

display(tab)